# 02 — Preprocessing (PROVIDED track)

Runs on the **de-identified published data only** — no raw JSON, no OpenAI API:

- `data/processed_20250904/app_selection_input_notext_20250904.json` — the collected app data with **review text removed** (only per-review timestamps/scores + app metadata). App selection reads review *counts* and *dates*, never review text, so this reproduces the identical selection.
- `data/processed_20250904/analysis_dataset.csv` — the PII-masked, analysis-ready reviews.

Outputs: reproduces the **app-selection funnel** and writes the **Step-3 input corpus** (`cleaned_reviews_of_categoryN.csv`). See README → *Provided track*.

In [1]:
import json
import pandas as pd
import numpy as np

In [2]:
date_str = "20250904"

In [3]:
# ---- Load the review-text-free app data (no raw JSON, no PII) ----
notext_path = f"../../data/processed_{date_str}/app_selection_input_notext_{date_str}.json"
with open(notext_path, encoding="utf-8") as f:
    data = json.load(f)
print(f"Apps loaded (no review text): {len(data)}")

# ---- Selection-filter constants (identical to 02_preprocess.ipynb) ----
RECENCY_CUTOFF = pd.Timestamp("2025-06-01")
MIN_REVIEWS    = 50
INSTALL_MIN    = 1_000
INSTALL_MAX    = 10_000_000

def get_latest_review_date(reviews):
    if not reviews:
        return pd.NaT
    dates = [pd.to_datetime(r.get("at"), errors="coerce")
             for r in reviews if isinstance(r, dict) and r.get("at") is not None]
    dates = [d for d in dates if pd.notna(d)]
    return max(dates) if dates else pd.NaT

def parse_install_bucket(s):
    if not isinstance(s, str):
        return None
    c = s.replace(",", "").replace("+", "").strip()
    return int(c) if c.isdigit() else None

df = pd.DataFrame([{
    "AppId":              d["appId"],
    "Title":              str(d["title"]),
    "Free":               d["free"],
    "Nb_Installs":        d["installs"],
    "Nb_Reviews":         len(d["reviews"]),          # count only - no text read
    "Latest_Review_Date": get_latest_review_date(d["reviews"]),
} for d in data])
df["Install_Bucket"] = df["Nb_Installs"].apply(parse_install_bucket)

eligible = (
    (df["Nb_Reviews"] >= MIN_REVIEWS)
    & (df["Latest_Review_Date"].notna())
    & (df["Latest_Review_Date"] >= RECENCY_CUTOFF)
    & (df["Free"] == True)
    & (df["Install_Bucket"].notna())
    & (df["Install_Bucket"] >= INSTALL_MIN)
    & (df["Install_Bucket"] <= INSTALL_MAX)
)
df["eligible"] = eligible
print(f"Total apps: {len(df)} | eligible: {int(eligible.sum())}")

Apps loaded (no review text): 1022


Total apps: 1022 | eligible: 388


In [4]:
# ---- Merge validated consumer/provider labels & print the selection funnel ----
val = pd.read_csv(f"../../data/processed_{date_str}/app_target_users_validated.csv")
gmap = {"1 Patients / Health-conscious Individuals": "HSC",
        "2 Healthcare Professionals / Medical Students": "HSP",
        "3 Other / Unclear Audience": "other"}
val["group"] = val["Target_Users"].map(gmap)
df = df.merge(val[["AppId", "group"]], on="AppId", how="left")

print("Selection funnel")
print(f"  collected apps           : {len(df)}")
print(f"  eligible (auto criteria) : {int(df['eligible'].sum())}")
print(f"  validated (classified)   : {int(val['group'].notna().sum())}")
print(f"  consumer (HSC)           : {int((df['group'] == 'HSC').sum())}")
print(f"  provider (HSP)           : {int((df['group'] == 'HSP').sum())}")
print(f"  other (excluded)         : {int((df['group'] == 'other').sum())}")

# Refresh the published app-level selection table
sel_cols = ["AppId", "Title", "group", "Nb_Reviews", "Latest_Review_Date", "Nb_Installs", "Free", "eligible"]
out = df.copy()
out["Latest_Review_Date"] = pd.to_datetime(out["Latest_Review_Date"]).dt.strftime("%Y-%m-%d")
out[sel_cols].to_csv(f"../../data/processed_{date_str}/app_selection_table.csv", index=False, encoding="utf-8")
print("Saved app_selection_table.csv")

Selection funnel
  collected apps           : 1022
  eligible (auto criteria) : 388
  validated (classified)   : 382
  consumer (HSC)           : 234
  provider (HSP)           : 91
  other (excluded)         : 57
Saved app_selection_table.csv


In [5]:
# ---- Build the Step-3 input corpus from the PII-masked analysis dataset ----
# analysis_dataset.csv already holds the analysis-ready reviews (masked text + rating
# + group). Split it back into the two category files that 03 consumes.
adf = pd.read_csv(f"../../data/processed_{date_str}/analysis_dataset.csv")
group_to_cat = {"HSC": "category1", "HSP": "category2"}
for grp, cat in group_to_cat.items():
    sub = adf[adf["group"] == grp]
    corpus = sub.rename(columns={"review": "content_clean", "rating": "score"})[["content_clean", "score"]]
    path = f"../../data/processed_{date_str}/cleaned_reviews_of_{cat}_provided.csv"
    corpus.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved {path}: {len(corpus)} reviews  (group={grp})")

print("")
print("Next: 03_sentiment_scoring_and_regression.py (optional; re-scores masked text),")
print("or 04_analysis_20250904.ipynb to reproduce the paper from analysis_dataset.csv directly.")

Saved ../../data/processed_20250904/cleaned_reviews_of_category1_provided.csv: 18929 reviews  (group=HSC)
Saved ../../data/processed_20250904/cleaned_reviews_of_category2_provided.csv: 6085 reviews  (group=HSP)

Next: 03_sentiment_scoring_and_regression.py (optional; re-scores masked text),
or 04_analysis_20250904.ipynb to reproduce the paper from analysis_dataset.csv directly.
